# W04 · 01 조합 규칙 3종 — 이종쌍 ε · σ 표
목표 3
1. `data/gaff2_atom_types.csv`(연구 플랫폼 GAFF2 결과 73분자 · 32타입) + `data/clayff_lj.csv`(Cygan 2004) 읽기
2. 산술(Lorentz–Berthelot · GAFF2) · 기하(OPLS) · 6제곱(Waldman–Hagler · COMPASS) 세 규칙으로 σ_ij · ε_ij 계산 → `out/mixing_table.csv`
3. 슬라이드 16 · 17 의 숫자 재현: ca–hc σ 2.958 / 2.936 / 3.058 Å · ca–Si(ClayFF) ε ≈ 4e−4

실행(기본): week04 폴더에서 연 Claude Code 에 「`@notebooks/01_mixing_rules.ipynb` 를 실행하고 결과를 요약해」 → 승인 → `python ../run_nb.py notebooks/01_mixing_rules.ipynb` · 그림을 직접 보려면 `jupyter lab`

In [ ]:
# 기본 = 내 컴퓨터 mdai 환경 · Claude Code 가 week04 폴더에서 실행 · 아래 google.colab 분기는 Colab(선택)에서만 동작
import os, sys, json
if "google.colab" in sys.modules and not os.path.exists("scripts/ff_tools.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week04
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from ff_tools import *   # load_types mix mixing_table cutoff_scan potential tail_correction OUT

## 1. 타입 표 읽기

In [ ]:
T = load_types()
print(len(T), "types")
for k in ["ca", "c3", "ha", "hc", "o", "ss", "O_clay", "Si_clay"]:
    print(f"{k:8s} ε {T[k][0]:.4g} kcal/mol · σ {T[k][1]:.4f} Å")

## 2. 세 규칙 (슬라이드 17)
- 산술(LB): σ_ij = (σ_i+σ_j)/2 · ε_ij = √(ε_i ε_j)
- 기하: σ_ij = √(σ_i σ_j) · ε_ij = √(ε_i ε_j)
- 6제곱(WH): σ_ij = [(σ_i⁶+σ_j⁶)/2]^(1/6) · ε_ij = 2√(ε_i ε_j) σ_i³σ_j³/(σ_i⁶+σ_j⁶)

In [ ]:
for a, b in [("ca", "hc"), ("ca", "O_clay"), ("ca", "Si_clay"), ("ca", "ca")]:
    m = mix(*T[a], *T[b])
    print(f"{a}-{b}: LB σ {m['lb'][0]:.3f} ε {m['lb'][1]:.4g} | geo σ {m['geo'][0]:.3f} | WH σ {m['wh'][0]:.3f} ε {m['wh'][1]:.4g}")
# 멱평균 부등식: 기하 ≤ 산술 ≤ 6제곱 — 모든 쌍에서 확인
rows = mixing_table(); bad = [r for r in rows if not (r["sigma_geo"] <= r["sigma_lb"] + 1e-9 <= r["sigma_wh"] + 1e-9)]
print(len(rows), "pairs →", "부등식 위반", len(bad))

## 3. 그림: ca–hc 세 규칙의 U_ij(r)

In [ ]:
a, b = T["ca"], T["hc"]; m = mix(*a, *b); r = np.linspace(2.4, 8, 400)
fig, ax = plt.subplots(figsize=(8, 3.8))
for k, name in [("lb", "arithmetic (LB)"), ("geo", "geometric"), ("wh", "sixth-power (WH)")]:
    s, e = m[k]; ax.plot(r, 4*e*((s/r)**12 - (s/r)**6), label=f"{name}: sigma {s:.3f}, eps {e:.4f}")
ax.set_ylim(-0.06, 0.06); ax.axhline(0, color="gray", lw=0.8); ax.set_xlabel("r (A)"); ax.set_ylabel("U (kcal/mol)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "mixing_ca_hc.png", dpi=120); plt.show()

## 확인 질문 (quiz.md)
1. ca–O(ClayFF) 이종쌍 ε 가 ca–Si(ClayFF) 보다 300배 큰 이유
2. 두 타입의 σ 가 같을 때 세 규칙의 σ_ij 가 같은 이유